In [1]:
import pandas as pd
from pathlib import Path

CLEAN_FILE = Path(
    r"F:\pojects for github\NLP-Based Customer Review & Complaint Analyzer"
    r"\review-analyzer\data\clean\swiggy_reviews_clean_ipynb.csv"
)

df = pd.read_csv(CLEAN_FILE)

print("Dataset loaded successfully.")
print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Shape:", df.shape)

Dataset loaded successfully.
Rows: 5000
Columns: 12
Shape: (5000, 12)


In [2]:
import torch
import transformers

print("PyTorch version:", torch.__version__)
print("Transformers version:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())

PyTorch version: 2.7.1+cpu
Transformers version: 4.53.0
CUDA available: False


In [3]:
from transformers import pipeline

sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    device=-1
)

print("DistilBERT sentiment model loaded successfully.")

Device set to use cpu


DistilBERT sentiment model loaded successfully.


In [4]:
test_reviews = [
    "The food was excellent and delivery was very fast.",
    "Worst app ever, my order was cancelled and I got no refund.",
    "The service was okay."
]

for review in test_reviews:
    result = sentiment_pipeline(review)[0]

    print("Review:", review)
    print("Sentiment:", result["label"])
    print("Confidence:", round(result["score"], 4))
    print("-" * 60)

Review: The food was excellent and delivery was very fast.
Sentiment: POSITIVE
Confidence: 0.9998
------------------------------------------------------------
Review: Worst app ever, my order was cancelled and I got no refund.
Sentiment: NEGATIVE
Confidence: 0.9998
------------------------------------------------------------
Review: The service was okay.
Sentiment: POSITIVE
Confidence: 0.9996
------------------------------------------------------------


In [5]:
from tqdm.auto import tqdm

reviews = df["content"].astype(str).tolist()

results = []

batch_size = 32

for i in tqdm(
    range(0, len(reviews), batch_size),
    desc="Processing reviews"
):
    batch = reviews[i:i + batch_size]

    batch_results = sentiment_pipeline(
        batch,
        truncation=True,
        max_length=512
    )

    results.extend(batch_results)

print("\nReviews processed:", len(results))

Processing reviews:   0%|          | 0/157 [00:00<?, ?it/s]


Reviews processed: 5000


In [6]:
df["sentiment"] = [result["label"] for result in results]
df["confidence"] = [result["score"] for result in results]

print("Sentiment columns added successfully.")

print("\nSentiment distribution:")
print(df["sentiment"].value_counts())

print("\nAverage confidence:",
      round(df["confidence"].mean(), 4))

Sentiment columns added successfully.

Sentiment distribution:
sentiment
POSITIVE    3011
NEGATIVE    1989
Name: count, dtype: int64

Average confidence: 0.9797


In [7]:
sentiment_rating = pd.crosstab(
    df["score"],
    df["sentiment"]
)

print("Sentiment vs Rating:")
print(sentiment_rating)

Sentiment vs Rating:
sentiment  NEGATIVE  POSITIVE
score                        
1              1479       108
2               106        43
3                69       144
4                59       386
5               276      2330


In [9]:
VALIDATION_FILE = Path(
    r"F:\pojects for github\NLP-Based Customer Review & Complaint Analyzer"
    r"\review-analyzer\data\processed\validation_50.csv"
)

validation_df = pd.read_csv(VALIDATION_FILE)

print("Validation reviews:", len(validation_df))
print("\nColumns:")
print(validation_df.columns.tolist())

print("\nManual sentiment distribution:")
print(validation_df["manual_sentiment"].value_counts())

Validation reviews: 50

Columns:
['reviewId', 'content', 'score', 'sentiment', 'manual_sentiment']

Manual sentiment distribution:
manual_sentiment
POSITIVE    23
NEGATIVE    23
UNCLEAR      4
Name: count, dtype: int64


In [10]:
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)

# Keep only manually labeled reviews
evaluation_df = validation_df[
    validation_df["manual_sentiment"] != "UNCLEAR"
].copy()

y_true = evaluation_df["manual_sentiment"]
y_pred = evaluation_df["sentiment"]

print("Labeled reviews:", len(evaluation_df))

print(
    "\nAccuracy:",
    round(accuracy_score(y_true, y_pred), 4)
)

print("\nConfusion matrix:")
print(
    confusion_matrix(
        y_true,
        y_pred,
        labels=["NEGATIVE", "POSITIVE"]
    )
)

print("\nClassification report:")
print(
    classification_report(
        y_true,
        y_pred,
        labels=["NEGATIVE", "POSITIVE"],
        digits=4
    )
)

Labeled reviews: 46

Accuracy: 0.9783

Confusion matrix:
[[22  1]
 [ 0 23]]

Classification report:
              precision    recall  f1-score   support

    NEGATIVE     1.0000    0.9565    0.9778        23
    POSITIVE     0.9583    1.0000    0.9787        23

    accuracy                         0.9783        46
   macro avg     0.9792    0.9783    0.9783        46
weighted avg     0.9792    0.9783    0.9783        46



In [11]:
errors = evaluation_df[
    evaluation_df["manual_sentiment"] != evaluation_df["sentiment"]
].copy()

print("Misclassified reviews:", len(errors))

print(
    errors[
        [
            "reviewId",
            "content",
            "score",
            "manual_sentiment",
            "sentiment"
        ]
    ].to_string(index=False)
)

Misclassified reviews: 1
                            reviewId                    content  score manual_sentiment sentiment
133b1b3b-7951-4869-8ed8-0581b8c0beed platform charge is to much      1         NEGATIVE  POSITIVE


In [12]:
print("Total reviews:", len(df))

print("\nMissing sentiment:", df["sentiment"].isna().sum())
print("Missing confidence:", df["confidence"].isna().sum())

print("\nSentiment values:")
print(df["sentiment"].value_counts())

print("\nConfidence statistics:")
print(df["confidence"].describe())

Total reviews: 5000

Missing sentiment: 0
Missing confidence: 0

Sentiment values:
sentiment
POSITIVE    3011
NEGATIVE    1989
Name: count, dtype: int64

Confidence statistics:
count    5000.000000
mean        0.979689
std         0.069974
min         0.504390
25%         0.998784
50%         0.999786
75%         0.999825
max         0.999887
Name: confidence, dtype: float64


In [13]:
from pathlib import Path

OUTPUT_DIR = Path(
    r"F:\pojects for github\NLP-Based Customer Review & Complaint Analyzer"
    r"\review-analyzer\data\processed"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SENTIMENT_FILE = OUTPUT_DIR / "sentiment_reviews_ipynb.csv"

df.to_csv(
    SENTIMENT_FILE,
    index=False,
    encoding="utf-8"
)

print("Sentiment dataset saved successfully.")
print("File:", SENTIMENT_FILE)
print("Rows saved:", len(df))
print("Columns saved:", len(df.columns))

Sentiment dataset saved successfully.
File: F:\pojects for github\NLP-Based Customer Review & Complaint Analyzer\review-analyzer\data\processed\sentiment_reviews_ipynb.csv
Rows saved: 5000
Columns saved: 14


In [14]:
saved_df = pd.read_csv(SENTIMENT_FILE)

print("Saved dataset shape:", saved_df.shape)

print("\nColumns:")
print(saved_df.columns.tolist())

print("\nMissing sentiment:",
      saved_df["sentiment"].isna().sum())

print("Missing confidence:",
      saved_df["confidence"].isna().sum())

print("\nSentiment distribution:")
print(saved_df["sentiment"].value_counts())

Saved dataset shape: (5000, 14)

Columns:
['reviewId', 'userName', 'userImage', 'content', 'score', 'thumbsUpCount', 'reviewCreatedVersion', 'at', 'replyContent', 'repliedAt', 'appVersion', 'detected_language', 'sentiment', 'confidence']

Missing sentiment: 0
Missing confidence: 0

Sentiment distribution:
sentiment
POSITIVE    3011
NEGATIVE    1989
Name: count, dtype: int64
